## Importar librerías y cargar API key de DeepSeek

In [ ]:
!pip install openai

In [ ]:
# Importación de librerías necesarias
from openai import OpenAI
from google.genai import types
from google.colab import userdata
import pandas as pd
import re
import tiktoken

# Cargar API key de DeepSeek desde variables seguras de Colab
DEEPSEEK_API_KEY  = userdata.get('DEEPSEEK_API_KEY ')
# Inicializar cliente OpenAI (DeepSeek es compatible con OpenAI)
client = OpenAI(api_key=DEEPSEEK_API_KEY , base_url="https://api.deepseek.com")
# Seleccionar version del modelo DeepSeek-V3.2 (cambia modo de razonamiento)
MODEL = "deepseek-chat"
# MODEL = "deepseek-reasoner"

## Cargar dataset

In [ ]:
# conectar Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Cargar datasets individuales por tarea
path = "/content/drive/MyDrive/tfg/corpusMentalRiskES/splits/LLM/"
df_anx = pd.read_csv(path + "anxiety_llm.csv", sep=",")
df_dep = pd.read_csv(path + "depression_llm.csv", sep=",")
df_ed = pd.read_csv(path + "ed_llm.csv", sep=",")
df_multi = pd.read_csv(path + "multiclass_llm.csv", sep=",")

# Seleccionar tarea de clasificacion: anx | dep | ed | multi
task = "anx" # <--- Cambiar aquí

# Asignar el dataframe correspondiente a la tarea seleccionada
if task == "anx":
  df = df_anx
elif task == "dep":
  df = df_dep
elif task == "ed":
  df = df_ed
elif task == "multi":
  df = df_multi

## Calcular número de tokens y coste

In [ ]:
def build_prompt(text, task):
  """ Construcción dinámica del prompt según tarea.
  Devuelve el texto completo que se enviará al modelo.
  """

  if task == "anx":
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede o no reflejar indicios de ansiedad.

La ansiedad se asocia a una preocupación persistente y difícil de controlar, acompañada de nerviosismo, inquietud interna y una sensación continua de amenaza o anticipación negativa. Estas experiencias suelen generar malestar emocional sostenido y dificultad para relajarse.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

1: el texto muestra indicios de ansiedad
0: el texto no muestra indicios de ansiedad

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente.
"""
  elif task == "dep":
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede o no reflejar indicios de depresión.

La depresión se asocia a un estado persistente de tristeza, apatía o vacío emocional, acompañado de una disminución del interés o la capacidad de disfrute en actividades habituales. Estas experiencias pueden incluir pensamientos negativos recurrentes, fatiga emocional y una percepción pesimista de uno mismo o del futuro.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

1: el texto muestra indicios de depresión
0: el texto no muestra indicios de depresión

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente.
"""
  elif task == "ed":
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede o no reflejar indicios de un trastorno de la conducta alimentaria.

Los trastornos de la conducta alimentaria se asocian a una preocupación persistente por el peso corporal, la alimentación o la imagen corporal, que puede manifestarse mediante pensamientos rígidos o negativos sobre la comida, el cuerpo o el control del peso. Estas preocupaciones suelen generar malestar emocional, culpa, ansiedad en torno a la alimentación y conductas de evitación o control.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

1: el texto muestra indicios de un trastorno de la conducta alimentaria
0: el texto no muestra indicios de un trastorno de la conducta alimentaria

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente.
"""
  else:
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede reflejar indicios de uno de los siguientes estados psicológicos, o no reflejar ninguno de ellos.

Ansiedad: se asocia a una preocupación persistente y difícil de controlar, acompañada de nerviosismo, inquietud interna y una sensación continua de amenaza o anticipación negativa, generando malestar emocional y dificultad para relajarse.

Depresión: se asocia a un estado persistente de tristeza, apatía o vacío emocional, junto con una disminución del interés o la capacidad de disfrute en actividades habituales, pensamientos negativos recurrentes y una visión pesimista de uno mismo o del futuro.

Trastornos de la conducta alimentaria: se asocian a una preocupación persistente por el peso, la alimentación o la imagen corporal, que puede manifestarse mediante pensamientos rígidos o negativos sobre la comida o el cuerpo, así como malestar emocional, culpa o ansiedad en torno a la alimentación.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

0: el texto no muestra indicios de ninguno de los estados descritos
1: el texto muestra indicios de ansiedad
2: el texto muestra indicios de depresión
3: el texto muestra indicios de un trastorno de la conducta alimentaria

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente.
"""

In [ ]:
# Obtener el tokenizador correspondiente al modelo
encoding = tiktoken.get_encoding("cl100k_base")

def total_tokens_dataset(df, task):
  """ Calcula el número total de tokens de entrada
  para un dataset completo, usando el tokenizador del modelo
  """
  total_tokens = 0

  for text in df["texto"]:
    prompt = build_prompt(text, task)
    tokens = len(encoding.encode(prompt))
    total_tokens += tokens

  return total_tokens


# Cálculo de tokens por dataset
total_anx = total_tokens_dataset(df_anx, "anx")
print("DATASET ANSIEDAD")
print("Tokens:", total_anx)
print("Media por texto:", total_anx / len(df_anx))

total_dep = total_tokens_dataset(df_dep, "dep")
print()
print("DATASET DEPRESIÓN")
print("Tokens:", total_dep)
print("Media por texto:", total_dep / len(df_dep))

total_ed = total_tokens_dataset(df_ed, "ed")
print()
print("DATASET ED")
print("Tokens:", total_ed)
print("Media por texto:", total_ed / len(df_ed))

total_multi = total_tokens_dataset(df_multi, "multi")
print()
print("DATASET MULTICLASE")
print("Tokens:", total_multi)
print("Media por texto:", total_multi / len(df_multi))

# Suma total de tokens necesarios para todo el experimento
print()
print(f"Total de tokens: {total_anx + total_dep + total_ed + total_multi}")

DATASET ANSIEDAD
Tokens: 511422
Media por texto: 1022.844

DATASET DEPRESIÓN
Tokens: 433676
Media por texto: 869.0901803607214

DATASET ED
Tokens: 299544
Media por texto: 894.1611940298508

DATASET MULTICLASE
Tokens: 1464529
Media por texto: 1097.8478260869565

Total de tokens: 2709171


## Clasificación zero-shot con DeepSeek

In [ ]:
def classify_text(text, task):
  """ Función de clasificación usando DeepSeek
  Devuelve:
  - int (0,1,2,3) si hay predicción válida
  - None si el modelo bloquea o no devuelve texto
  """
  try:
    response = client.chat.completions.create(
      model=MODEL,
      messages=[
        {"role": "user", "content": build_prompt(text, task)}
      ],
      temperature=0,
      max_tokens=5
    )

    output_text = response.choices[0].message.content

    # Devuelve None si no hay texto
    # (suele ocurrir porque bloquea el prompt por PROHIBITED_CONTENT)
    if not output_text:
      return None

    # Extraer número de clase mediante expresión regular
    match = re.search(r"[0-3]", output_text)

    return int(match.group()) if match else None
  except Exception:
    return None

In [ ]:
# Aplicar clasificación al dataset seleccionado
# Se almacena la predicción en una nueva columna "pred"
df["pred"] = df["texto"].apply(lambda x: classify_text(x, task))

### Comprobación de predicciones

In [ ]:
from sklearn.metrics import f1_score

# Columna con la etiqueta real
true_labels = df["bs"]
# Reemplazar None por una clase inválida para que cuente como error
pred_labels = df["pred"].fillna(-1)

# Si es clasificación binaria
if task in ["anx", "dep", "ed"]:
    f1 = f1_score(true_labels, pred_labels, average="binary")
else:
    # Clasificación multiclase
    f1 = f1_score(true_labels, pred_labels, average="macro")

print("F1-score:", round(f1, 4))

In [ ]:
# Número de bloqueos
num_blocked = df["pred"].isna().sum()
block_rate = num_blocked / len(df)

print("Bloqueos:", num_blocked)
print("Tasa de bloqueo:", round(block_rate, 4))

### Almacenar predicciones

In [ ]:
output_path = (
    f"/content/drive/MyDrive/tfg/corpusMentalRiskES/resultados/"
    f"Gemini_{MODEL}_{task}.csv"
)
df.to_csv(output_path, index=False)